<a href="https://colab.research.google.com/github/haridevsreebhavan/AI-Training-/blob/main/Assignments/Assignment_6/Assignment_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain

In [2]:
!pip install -U transformers accelerate sentence-transformers faiss-cpu

In [3]:
!pip install -U langchain langchain-community pypdf

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/intro-to-ml.pdf")
documents = loader.load()

In [5]:
print(f"Total Pages: {len(documents)}")
print(documents[0].page_content[:500])

Total Pages: 392
Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR DATA SCIENTISTS


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

Total chunks: 1137


In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_23696/384917042.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embedding_model)
vectorstore.save_local("faiss_index")

In [9]:
query = "What is supervised learning?"
docs = vectorstore.similarity_search(query, k=3)

for doc in docs:
    print(doc.page_content[:300])

CHAPTER 2
Supervised Learning
As we mentioned earlier, supervised machine learning is one of the most commonly
used and successful types of machine learning. In this chapter, we will describe super‐
vised learning in more detail and explain several popular supervised learning algo‐
rithms. We alread
2. Supervised Learning. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .  25
Classification and Regression                                                                                         25
Generalization, Overfitting, and Underfit
chapter, we will go into more depth about the different kinds of supervised models in
scikit-learn and how to apply them successfully.
24 | Chapter 1: Introduction


In [10]:
def retrieve_docs(query, k=5):
    return vectorstore.similarity_search(query, k=k)

In [12]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM',

In [13]:
def generate_answer(query, docs):
    context = "\n\n".join([d.page_content for d in docs])

    prompt = f"""
You are a Machine Learning expert.

Answer ONLY using the context below.
If the answer is not present, say "I don't know".

Context:
{context}

Question:
{query}

Answer:
"""

    result = llm(prompt, max_new_tokens=150)
    return result[0]['generated_text']

In [14]:
def rag_pipeline(query):
    docs = vectorstore.similarity_search(query, k=5)
    return generate_answer(query, docs)

In [ ]:
while True:
    query = input("\nAsk a question: ")

    if query.lower() == "exit":
        break

    print(rag_pipeline(query))


Ask a question: What is overfitting?


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (837 > 512). Running this sequence through the model will result in indexing errors
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a Machine Learning expert.

Answer ONLY using the context below.
If the answer is not present, say "I don't know".

Context:
on a single customer.
The only measure of whether an algorithm will perform well on new data is the eval‐
uation on the test set. However, intuitively 3 we expect simple models to generalize
better to new data. If the rule was “People older than 50 want to buy a boat, ” and this
would explain the behavior of all the customers, we would trust it more than the rule
involving children and marital status in addition to age. Therefore, we always want to
find the simplest model. Building a model that is too complex for the amount of
information we have, as our novice data scientist did, is called overfitting. Overfitting
occurs when you fit a model too closely to the particularities of the training set and
obtain a model that works well on the training set but is not able to generalize to new

Out[28]:
Training set score: 0.67
Test set score: 0.66
An R2 of aro

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a Machine Learning expert.

Answer ONLY using the context below.
If the answer is not present, say "I don't know".

Context:
these algorithms to the built-in datasets in scikit-learn, like the boston_housing or
diabetes datasets for regression, or the digits dataset for multiclass classification.
Playing around with the algorithms on different datasets will give you a better feel for
128 | Chapter 2: Supervised Learning

To assess the model’s performance, we show it new data (data that it hasn’t seen
before) for which we have labels. This is usually done by splitting the labeled data we
have collected (here, our 150 flower measurements) into two parts. One part of the
data is used to build our machine learning model, and is called the training data or
training set. The rest of the data will be used to assess how well the model works; this
is called the test data, test set, or hold-out set.
scikit-learn contains a function that shuffles the dataset and splits it for you: the
tr